In [1]:
!pip install -U transformers datasets rouge-score accelerate tensorboard

import torch
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from datasets import load_dataset
from rouge_score import rouge_scorer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 15.0 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=92715fd0bd86ded0b28de676e617b9b042a768507f7f27aad89c61bdf5c06266
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.19.0
    Uninstalling t

In [3]:
!pwd

/content


In [6]:
# ======================
# Config
# ======================

MODEL_CHECKPOINT = "google-t5/t5-small"   # or t5-base, t5-large
MAX_INPUT_LENGTH = 2048
MAX_TARGET_LENGTH = 128
BATCH_SIZE = 8
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
OUTPUT_DIR = "./t5-summarization-model"
CSV_PATH = "tldr.csv"

prefix = "summarize: "   # or "slangify: " etc.

# ======================
# Load dataset from single CSV and split
# ======================

data_files = {"data": CSV_PATH}
dataset = load_dataset("csv", data_files=data_files)

# 90% train, 10% validation
dataset = dataset["data"].train_test_split(test_size=0.1, seed=42)
dataset["validation"] = dataset["test"]
del dataset["test"]

# ======================
# Tokenizer & model
# ======================

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model parameters: 60,506,624


In [26]:
dataset['train'][0]

{'prompt': "SUBREDDIT: r/AskReddit TITLE: My mom hates my girlfriend's parents and is forbidding me from seeing her. I'm stuck reddit. POST: My girlfriend and I have been dating for a little over a year now and things generally have gone amazing. A few months in however when I was at her parents cabin we were all having dinner and her dad offered me a beer (I'm sixteen) I declined and nothing happened after that and I thought nothing of it. When I got home it somehow casually slipped into conversation and my mother flipped. I calmed her down about it and told her I didn't take it I'm a good kid etc etc. a few months later we were fooling around in her bedroom when her father walked in on us and we both were in major trouble to the point where we weren't allowed to speak to each other for a month. This obviously put a big strain on her parents and my mom (my father's deceased) in the trust department. Eventually things became a little more normal after. About a month ago however while i

In [70]:
# ======================
# Preprocessing
# ======================

def preprocess_function(examples):
    """
    Uses:
      - input:  examples["prompt"], examples["completion"]
      - target: examples["GenZ_completion"]
    """

    if None in examples["GenZ_completion"]:
      return

    inputs = [prefix + doc for doc in examples["prompt"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        # padding="max_length"
    )

    # Targets
    targets = examples["GenZ_completion"]

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
            # padding="max_length"
        )


    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=dataset["train"].column_names
)

Map (num_proc=4):   0%|          | 0/103080 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/11454 [00:00<?, ? examples/s]

In [60]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion', 'top_k_slang', 'GenZ_completion'],
        num_rows: 103080
    })
    validation: Dataset({
        features: ['prompt', 'completion', 'top_k_slang', 'GenZ_completion'],
        num_rows: 11454
    })
})

In [51]:
# ======================
# Metrics (ROUGE)
# ======================

rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"], use_stemmer=True
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Decode predictions
    pred_ids = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)

    # Replace -100 in labels as well
    labels_ids = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels_ids, skip_special_tokens=True)

    rouge1_list, rouge2_list, rougeL_list = [], [], []
    for pred, label in zip(decoded_preds, decoded_labels):
        scores = rouge.score(label, pred)
        rouge1_list.append(scores["rouge1"].fmeasure)
        rouge2_list.append(scores["rouge2"].fmeasure)
        rougeL_list.append(scores["rougeL"].fmeasure)

    return {
        "rouge1": np.mean(rouge1_list),
        "rouge2": np.mean(rouge2_list),
        "rougeL": np.mean(rougeL_list),
    }

In [43]:
# ======================
# Training setup
# ======================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    # evaluation_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=100,
    save_steps=500,
    eval_steps=500,
    save_total_limit=3,
    # load_best_model_at_end=True,
    metric_for_best_model="rouge1",
    greater_is_better=True,
    fp16=True,
    # predict_with_generate=True,
    report_to="tensorboard",
    seed=42,
    remove_unused_columns=False
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    return_tensors="pt"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

/tmp/ipython-input-2474598618.py:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [44]:
# ======================
# Train
# ======================

history = trainer.train()

# Save final model & tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Training complete! Model saved to {OUTPUT_DIR}")

ValueError: You should supply an encoding or a list of encodings to this method that includes input_ids, but you provided ['prompt', 'completion', 'top_k_slang', 'GenZ_completion']

In [ ]:
# ======================
# Inference example
# ======================

fine_tuned_model = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR)
fine_tuned_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

fine_tuned_model.to(device)

# Example prompt (same style as your 'prompt' column)
input_text = "Your input text here."

inputs = fine_tuned_tokenizer(
    prefix + input_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
).to(device)

summary_ids = fine_tuned_model.generate(
    inputs["input_ids"],
    max_length=128,
    min_length=20,
    num_beams=4,
    early_stopping=True,
    no_repeat_ngram_size=2
)

output_text = fine_tuned_tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)
print(f"Model output: {output_text}")